In [ ]:

# =====================================================
# ASL SIGN LANGUAGE - FINAL ONE CELL NOTEBOOK
# =====================================================

import os
import zipfile
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

DATASET_PATH = "dataset/asl_alphabet_train"

# =====================================================
# CEK / DOWNLOAD DATASET
# =====================================================

if not os.path.exists(DATASET_PATH):

    print("Dataset tidak ditemukan. Mengunduh dari Kaggle...")

    os.makedirs("dataset", exist_ok=True)

    status = os.system(
        "kaggle datasets download grassknoted/asl-alphabet -p dataset"
    )

    if status != 0:
        raise Exception(
            "Gagal download dataset. Pastikan Kaggle API sudah terpasang dan kaggle.json sudah dikonfigurasi."
        )

    with zipfile.ZipFile(
        "dataset/asl-alphabet.zip",
        "r"
    ) as zip_ref:
        zip_ref.extractall("dataset")

    print("Dataset berhasil diunduh.")

else:
    print("Dataset ditemukan.")

# =====================================================
# LOAD DATASET
# =====================================================

IMG_SIZE = (224,224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names

with open("labels.txt","w",encoding="utf-8") as f:
    for label in class_names:
        f.write(label + "\n")

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =====================================================
# MODEL
# =====================================================

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

model = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.Rescaling(1./255),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation="relu"),
    layers.Dense(len(class_names), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)

# =====================================================
# TRAIN
# =====================================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

# =====================================================
# EVALUATE
# =====================================================

loss, accuracy = model.evaluate(val_ds)

print(f"Final Accuracy: {accuracy:.4f}")

# =====================================================
# EXPORT MODEL
# =====================================================

model.save("sign_language_model.keras")

print("Model berhasil disimpan.")
print("File model : sign_language_model.keras")
print("File label : labels.txt")

# =====================================================
# VISUALISASI
# =====================================================

plt.figure(figsize=(10,5))
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()


Dataset ditemukan.
Found 87000 files belonging to 1 classes.
Using 69600 files for training.
Found 87000 files belonging to 1 classes.
Using 17400 files for validation.
Epoch 1/10


c:\Users\Titan\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
